## Add/Update FMM Levels

- read file from folder
- convert YAML or JSON to dict
  - need to preserve my yaml comments  (try ruaml)
- add fmm extension if missing
  - if extension present then append new extension else create extension
  - get FMM level from csv file
  - if already has fmm extension then update fmm level
  - get FMM level from csv file
- validate
- save file back to folder

YAML:

- StuctureDefinition = see table
- OperationDefinition = 5
- CodeSysytem FMM = 3
- ValueSet = FMM= 3

JSON: 

- SearchParameter = FMM 5 
- CapabilityStatement = FMM 3

In [ ]:
from pathlib import Path
from json import dumps, loads
from fhir.resources.valueset import ValueSet
import io
from ruamel.yaml import YAML
import pandas

# Initialize ruamel.yaml in "round-trip" mode (preserving comments)
yaml = YAML(typ='rt')
yaml.indent(mapping=2, sequence=4, offset=2)
yaml.width = 4096

### globals

In [ ]:
in_path_yaml = Path(r'/Users/ehaas/Documents/FHIR/US-Core/input/resources-yaml')
in_path = Path(r'/Users/ehaas/Documents/FHIR/US-Core/input/resources')
# out_path = Path(r'/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/utils/out/resources') #for testing
# out_path_yaml = Path(r'/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/utils/out/resources') #for testing
out_path = in_path
out_path_yaml = in_path_yaml

fmm_types =  [('OperationDefinition',5,'yaml'),('searchparameter',5,'json'),('ValueSet',3,'yaml'),('CodeSystem',3,'yaml'),('capabilitystatement',3,'json'),("StructureDefinition",5,'yaml')]  # (type,fmm,sourcepath)
fmm_path = Path(r'/Users/ehaas/Documents/FHIR/US-Core/input/data/dstu2-r4-table.csv')
df = pandas.read_csv(fmm_path)
fmm_url = 'http://hl7.org/fhir/StructureDefinition/structuredefinition-fmm'

### Get FMM Levels for StructureDefinitions

- fetch CSV file
- convert to Dataframe

In [ ]:
def get_fmm_from_csv(profile_title):
  # print(profile_title)
  profile_fmm = df.loc[df['Equivalent US Core Profile'] == profile_title, 'proposed_fmm'].iloc[0]
  return int(profile_fmm)

### Create fmm level function

- create dict of extension
- insert fmm level as parameter

In [ ]:
def fmm_ext(level=5):

  return {'url': 'http://hl7.org/fhir/StructureDefinition/structuredefinition-fmm', 'valueInteger': level}

### get conformance resources from local directory

In [ ]:
def process_json_file(obj,fmm = 5):
    # print(obj.name)
    fhir_obj = loads(obj.read_text())
    # print(fhir_obj['id'])
    try:
      old_fmm_ext = [i for i,ext in enumerate(fhir_obj['extension']) if ext['url'] == fmm_url]
      if old_fmm_ext:
          #update the fmm
          fhir_obj['extension'][i]['valueInteger']=fmm
      else:
          fhir_obj["extension"].append(fmm_ext(fmm))
    except KeyError:
        fhir_obj["extension"] = [fmm_ext(fmm)]

    out_file = out_path / obj.name
    out_file.write_text(dumps(fhir_obj, sort_keys=False, indent=2))
    return

In [ ]:
def process_yaml_file(obj,fmm = 5):
    # print(obj.name)
    fhir_obj = yaml.load(obj.read_text())
    if fhir_obj['resourceType'] == 'StructureDefinition':  # get fmm from csv
        fmm = get_fmm_from_csv(fhir_obj['title'])
    # print(fhir_obj['id'])
    try:
      # print(fhir_obj['extension'])
      fmm_index = [i for i,ext in enumerate(fhir_obj['extension']) if ext['url'] == fmm_url]
      if fmm_index: #update the fmm
          # print(fmm_index,  fhir_obj['extension'][fmm_index[0]], fhir_obj['extension'][fmm_index[0]]['valueInteger'])
          fhir_obj['extension'][fmm_index[0]]['valueInteger']=fmm
      else:  #add the fmm
          fhir_obj["extension"].append(fmm_ext(fmm))
    except KeyError:
        fhir_obj["extension"] = [fmm_ext(fmm)]
    out_file = out_path_yaml / obj.name
    # the ruamel.yaml library requires a stream argument for its dump method
    stream = io.StringIO()
    yaml.dump(fhir_obj, stream)
    out_file.write_text(stream.getvalue())
    return

In [ ]:
for fhir_type in fmm_types:
  print(f'fhir type = {fhir_type[0]}')
  count = 0
  if fhir_type[2] == "json":
    for i, obj in enumerate(in_path.glob(f"{fhir_type[0]}-*.json")):
        process_json_file(obj,fmm = fhir_type[1])
        count += 1
  if fhir_type[2] == "yaml":
    for i, obj in enumerate(in_path_yaml.glob(f"{fhir_type[0]}-*.yml")):
        process_yaml_file(obj,fmm = fhir_type[1])
        count += 1
  print(f'{count} {fhir_type[0]} files written to {out_path}')
